## 4.1 Prompt Básico

O prompt mais simples possível: forneça o contexto e faça a pergunta.

Funciona para casos simples mas tem um problema grave: o LLM não recebe instruções explícitas sobre o que fazer quando a resposta não está no contexto. A tendência natural do LLM é completar o texto de forma coerente — o que significa *inventar* informações plausíveis.

Este comportamento (alucinação) pode parecer correto à primeira vista porque o LLM é convincente. É o maior risco em sistemas RAG de produção.

O exemplo abaixo mostra como um prompt básico se comporta:

In [ ]:
import sys
sys.path.insert(0, '..')

import ollama
import httpx

try:
    r = httpx.get('http://localhost:11434/api/tags')
    modelos = [m['name'] for m in r.json().get('models', [])]
    LLM = 'llama3.2' if any('llama3.2' in m for m in modelos) else (modelos[0] if modelos else None)
    print(f'LLM: {LLM}')
    print(f'Todos: {modelos}')
except Exception as e:
    LLM = None
    print(f'Ollama offline: {e}')

# Contexto de exemplo para testes
contexto_exemplo = """
[Fonte: vector_databases_overview.md]
HNSW (Hierarchical Navigable Small World) e o algoritmo de indexacao mais popular para ANN.
Os parametros principais sao:
- m: numero de conexoes por no (default 16)
- ef_construct: candidatos durante construcao (default 100)
- ef: candidatos durante busca

[Fonte: rag_fundamentals.md]
Para RAG de producao, recomenda-se usar int8 Scalar Quantization no Qdrant.
Isso reduz memoria em 75% com apenas 1-3% de perda de recall.
O parametro rescore=True recupera a precisao original.
"""

print('Pronto!')

## 4.1 Prompt Básico

O prompt mais simples possível: forneça o contexto e faça a pergunta.

Funciona para casos simples mas tem um problema grave: o LLM não recebe instruções explícitas sobre o que fazer quando a resposta não está no contexto. A tendência natural do LLM é completar o texto de forma coerente — o que significa *inventar* informações plausíveis.

Este comportamento (alucinação) pode parecer correto à primeira vista porque o LLM é convincente. É o maior risco em sistemas RAG de produção.

O exemplo abaixo mostra como um prompt básico se comporta:

In [ ]:
## 4.2 Prompt com Grounding Explícito

A solução para alucinação: instrua explicitamente o LLM a responder APENAS com base no contexto fornecido.

A adição de frases como "Responda SOMENTE com base nas informações fornecidas" e "Se a resposta não estiver no contexto, diga 'Não encontrei essa informação'" reduz drasticamente a taxa de alucinação.

**Por que funciona?** LLMs são treinados para seguir instruções. Quando você explicita o comportamento esperado, o modelo ajusta seu output. A instrução deve ser clara, direta e específica — instruções vagas ("use o contexto") são menos efetivas que instruções precisas ("responda SOMENTE com as informações do contexto, sem adicionar nada que não esteja lá").

In [ ]:
pergunta = 'Quais sao os parametros do HNSW e o que cada um faz?'

print(f'Pergunta: {pergunta}')
print('='*60)

for nome in PROMPTS:
    print(f'\n--- Template: {nome} ---')
    resposta = gerar(pergunta, nome)
    print(resposta[:500])
    if len(resposta) > 500:
        print('...(truncado)')

## 4.2 Alucinacao: Quando o LLM vai alem do contexto

In [ ]:
## 4.2 Prompt com Grounding Explícito

A solução para alucinação: instrua explicitamente o LLM a responder APENAS com base no contexto fornecido.

A adição de frases como "Responda SOMENTE com base nas informações fornecidas" e "Se a resposta não estiver no contexto, diga 'Não encontrei essa informação'" reduz drasticamente a taxa de alucinação.

**Por que funciona?** LLMs são treinados para seguir instruções. Quando você explicita o comportamento esperado, o modelo ajusta seu output. A instrução deve ser clara, direta e específica — instruções vagas ("use o contexto") são menos efetivas que instruções precisas ("responda SOMENTE com as informações do contexto, sem adicionar nada que não esteja lá").

## 4.3 Streaming de Respostas

In [ ]:
if LLM:
    print('Resposta com streaming:')
    print('-'*40)
    
    prompt = PROMPTS['with_citation'].format(
        contexto=contexto_exemplo,
        pergunta='Como reduzir o uso de memoria do Qdrant em producao?'
    )
    
    stream = ollama.chat(
        model=LLM,
        messages=[{'role': 'user', 'content': prompt}],
        stream=True,
    )
    
    for chunk in stream:
        print(chunk['message']['content'], end='', flush=True)
    print('\n' + '-'*40)
else:
    print('Ollama offline — streaming nao disponivel')

## 4.4 Context Stuffing: Qual e o limite?

Cada modelo tem um limite de contexto. Com muitos chunks, o LLM pode se perder.

In [ ]:
## 4.2 Prompt com Grounding Explícito

A solução para alucinação: instrua explicitamente o LLM a responder APENAS com base no contexto fornecido.

A adição de frases como "Responda SOMENTE com base nas informações fornecidas" e "Se a resposta não estiver no contexto, diga 'Não encontrei essa informação'" reduz drasticamente a taxa de alucinação.

**Por que funciona?** LLMs são treinados para seguir instruções. Quando você explicita o comportamento esperado, o modelo ajusta seu output. A instrução deve ser clara, direta e específica — instruções vagas ("use o contexto") são menos efetivas que instruções precisas ("responda SOMENTE com as informações do contexto, sem adicionar nada que não esteja lá").

## 4.2 Prompt com Grounding Explícito

A solução para alucinação: instrua explicitamente o LLM a responder APENAS com base no contexto fornecido.

A adição de frases como "Responda SOMENTE com base nas informações fornecidas" e "Se a resposta não estiver no contexto, diga 'Não encontrei essa informação'" reduz drasticamente a taxa de alucinação.

**Por que funciona?** LLMs são treinados para seguir instruções. Quando você explicita o comportamento esperado, o modelo ajusta seu output. A instrução deve ser clara, direta e específica — instruções vagas ("use o contexto") são menos efetivas que instruções precisas ("responda SOMENTE com as informações do contexto, sem adicionar nada que não esteja lá").

## Resumo: Evolução do Prompt

| Versão | Alucinação | Rastreabilidade | Complexidade |
|--------|-----------|-----------------|-------------|
| Básico | Alta | Nenhuma | Mínima |
| Com grounding | Baixa | Nenhuma | Baixa |
| Com citações | Baixa | Alta | Média |
| Estruturado | Baixíssima | Alta | Média-alta |

**Template de produção recomendado:**
```
Você é um assistente especializado. Responda APENAS com base nos documentos abaixo.
Se a resposta não estiver nos documentos, diga: "Não encontrei essa informação."

DOCUMENTOS:
[1] {doc1}
[2] {doc2}
...

PERGUNTA: {query}

RESPOSTA (cite as fontes como [1], [2] etc.):
```

**Próximos passos:**
- [01 — Naive RAG Architecture](../04_rag_architectures/01_naive_rag_arch.html): como essas peças se combinam numa arquitetura de produção?